In [12]:
import requests
import random
import time
import json

def login():
    url = "http://localhost:32677/api/v1/users/login"
    cookies = {
        'JSESSIONID': '9ED5635A2A892A4BA31E7E98533A279D',
        'YsbCaptcha': '025080CF8BA94594B09E283F17815444',
    }
    headers = {
        'Accept': 'application/json, text/javascript, */*; q=0.01',
        'X-Requested-With': 'XMLHttpRequest',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.107 Safari/537.36',
        'Content-Type': 'application/json',
        'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
    }
    data = '{"username":"fdse_microservice","password":"111111"}'
    
    response = requests.post(url, headers=headers, cookies=cookies, data=data)
    if response.status_code == 200:
        data = response.json().get("data")
        return data.get("userId"), data.get("token")
    return None, None

def query_contacts(headers, uuid):
    url = f"http://localhost:32677/api/v1/contactservice/contacts/account/{uuid}"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json().get("data")
        return [d.get("id") for d in data if d.get("id") is not None]
    return None

def query_high_speed_ticket(headers, place_pair=("Shang Hai", "Su Zhou")):
    url = "http://localhost:32677/api/v1/travelservice/trips/left"
    payload = {
        "departureTime": time.strftime("%Y-%m-%d"),
        "startingPlace": place_pair[0],
        "endPlace": place_pair[1],
    }
    
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        data = response.json().get("data")
        return [d.get("tripId").get("type") + d.get("tripId").get("number") for d in data]
    return None

def query_and_preserve():
    # 1. Login
    uuid, token = login()
    if not token:
        print("Login failed")
        return
    
    headers = {
        "Cookie": "JSESSIONID=823B2652E3F5B64A1C94C924A05D80AF; YsbCaptcha=2E037F4AB09D49FA9EE3BE4E737EAFD2",
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    # 2. Query contacts
    contacts = query_contacts(headers, uuid)
    if not contacts:
        print("No contacts found")
        return
    contact_id = random.choice(contacts)
    
    # 3. Query available trips
    trip_ids = query_high_speed_ticket(headers)
    if not trip_ids:
        print("No trips available")
        return
    trip_id = random.choice(trip_ids)
    
    # 4. Prepare preserve payload
    preserve_url = "http://localhost:32677/api/v1/preserveservice/preserve"
    preserve_data = {
        "accountId": uuid,
        "assurance": "0",
        "contactsId": contact_id,
        "date": time.strftime("%Y-%m-%d"),
        "from": "Shang Hai",
        "to": "Su Zhou",
        "tripId": trip_id,
        "foodType": "0",
        "seatType": random.choice(["2", "3"]),  # Random seat type
        "consigneeName": "Test User",
        "consigneePhone": "1234567890",
        "consigneeWeight": random.randint(1, 10),
        "handleDate": time.strftime("%Y-%m-%d")
    }
    
    # 5. Make preserve request
    response = requests.post(preserve_url, headers=headers, json=preserve_data)
    if response.status_code == 200 and response.json().get("data") == "Success":
        print("Preserve successful!")
        print(f"Trip ID: {trip_id}")
        print(f"Order details: {response.json()}")
    else:
        print("Preserve failed")
        print(response.text)

if __name__ == "__main__":
    query_and_preserve()

Preserve failed
{"timestamp":1743858974508,"status":500,"error":"Internal Server Error","exception":"org.springframework.web.client.HttpServerErrorException","message":"500 Internal Server Error","path":"/api/v1/preserveservice/preserve"}


In [15]:
query_and_preserve()

Preserve successful!
Trip ID: D1345
Order details: {'status': 1, 'msg': 'Success.', 'data': 'Success'}
